# VideoFlexTok: Reconstruction Quality vs. Token Budget

## Project overview

This notebook documents a small, independent inference experiment using the
official pretrained VideoFlexTok model. It is **not** a full reproduction of
the paper, and the model was not trained or fine-tuned.

### Research question

How does the number of retained tokens relate to pixel-level reconstruction
error for a small set of video clips?

### Experimental setup

- Model: `EPFL-VILAB/videoflextok_d18_d18_k600`
- Input: 17 frames resized to 128 × 128
- Hardware: Google Colab Tesla T4
- Decoder: 40 timesteps, guidance scale 25, seed 42
- Metrics: mean absolute error (MAE) and mean squared error (MSE)
- Results: saved CSV files and visualizations in `results/`

The model's decoding process is expensive, so this notebook analyzes saved
outputs instead of rerunning inference automatically.


## 1. Environment

The versions below were recorded from the Colab runtime used for the experiment.
A matching package list helps document the setup, but does not guarantee that
the experiment will run unchanged in every future runtime.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = Path.cwd()

# If the notebook is opened from the project directory, this works directly.
# Otherwise, set PROJECT to the project's github-project directory.
if not (PROJECT / "environment.json").exists():
    candidates = [
        Path("/content/drive/MyDrive/VideoFlexTok-Experiment/github-project"),
        Path("/content/VideoFlexTok-Experiment/github-project"),
    ]
    PROJECT = next(
        (p for p in candidates if (p / "environment.json").exists()),
        PROJECT,
    )

RESULTS = PROJECT / "results"
environment = json.loads((PROJECT / "environment.json").read_text())
print(json.dumps(environment, indent=2))


## 2. Token-budget results

The table below loads the measurements from the saved CSV. Errors are computed
in the experiment's normalized pixel space; they should not be interpreted as
perceptual quality scores.


In [ ]:
budget_results = pd.read_csv(RESULTS / "token_budget_results.csv")
display(budget_results)


In [ ]:
# Plot MAE and MSE against retained-token budget.
# Inspect the column names printed above if adapting this cell to a changed CSV.
print("Available columns:", budget_results.columns.tolist())


## 3. Saved error plot

This is the visualization generated during the experiment.


In [ ]:
from IPython.display import display, Image

plot_path = RESULTS / "token_budget_error.png"
if plot_path.exists():
    display(Image(filename=str(plot_path)))
else:
    print(f"Plot not found: {plot_path}")


## 4. Comparison across two clips

The second CSV contains selected measurements for the Porsche clip and a
second clip. The second clip was evaluated at 8 and 32 tokens only, so this
comparison does not establish a general result across videos.


In [ ]:
second_results = pd.read_csv(RESULTS / "second_clip_results.csv")
display(second_results)


## 5. Qualitative reconstruction

The image below compares the original Porsche clip's middle frame with the
middle frame from a 32-token reconstruction. It is one illustrative frame,
not a complete temporal evaluation.


In [ ]:
comparison_path = RESULTS / "original_vs_32_tokens.png"
if comparison_path.exists():
    display(Image(filename=str(comparison_path)))
else:
    print(f"Image not found: {comparison_path}")


## 6. Interpretation

In the recorded measurements, increasing the retained-token budget is
associated with lower pixel reconstruction error for the tested Porsche clip.
The second clip also has lower error at 32 tokens than at 8 tokens.

These observations are consistent with the idea that retaining more tokens can
preserve more reconstruction information in these examples. They do **not**
prove that the same trend or magnitude holds for arbitrary videos.

### Limitations

- Only two clips were tested.
- The second clip has measurements at only 8 and 32 tokens.
- The 1, 2, 4, and 8-token Porsche measurements were recorded in an earlier
  runtime; the 16 and 32-token measurements were recorded after a runtime reset.
- Pixel-level MAE/MSE do not fully capture perceptual quality or temporal
  consistency.
- No repeated-seed study, downstream-task evaluation, or model training was
  performed.
- Decoder settings and preprocessing affect the results.

### Reproducibility and attribution

The notebook analyzes saved outputs and does not reproduce inference from
scratch. For a full rerun, consult the official repository for setup and model
usage instructions:

- https://github.com/apple/ml-videoflextok
- https://videoflextok.epfl.ch/

The pretrained checkpoint is hosted as
`EPFL-VILAB/videoflextok_d18_d18_k600`. Check the upstream repository's license
and citation requirements before redistributing code or model assets.

### Conclusion

This is a preliminary, small-scale analysis of reconstruction error versus
token budget using a pretrained model. Broader claims require more clips,
consistent repeated runs, and perceptual or downstream evaluations.
